# 🫧 Lesson 8: Bubble Pop!

Py blew a bunch of bubbles and they won't stop floating around! Click on
one to pop it, and a new bubble appears somewhere else. How high can you
get your score?

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join("..", "..")))
from helpers.kid_tools import celebrate, try_again, check_answer, quiz, badge


## How this works

Each bubble is just a Python dictionary remembering its `x`, `y`,
`radius`, and `color`. When you click, your code checks how far the
click was from the *center* of each bubble — close enough, and it pops!

Run the cell below to start playing.

In [4]:
from ipycanvas import Canvas
import ipywidgets as widgets
from IPython.display import display
import math
import random

WIDTH, HEIGHT = 480, 360
BUBBLE_RADIUS = 22
NUM_BUBBLES = 5
COLORS = ["#e63946", "#f1c40f", "#2a9d8f", "#457b9d", "#e76f51", "#8e44ad"]

canvas = Canvas(width=WIDTH, height=HEIGHT)
bubbles = []
score = 0
status = widgets.Output()


def spawn_bubble():
    return {
        "x": random.randint(BUBBLE_RADIUS, WIDTH - BUBBLE_RADIUS),
        "y": random.randint(BUBBLE_RADIUS, HEIGHT - BUBBLE_RADIUS),
        "r": BUBBLE_RADIUS,
        "color": random.choice(COLORS),
    }


def redraw():
    canvas.clear()
    for b in bubbles:
        canvas.fill_style = b["color"]
        canvas.fill_circle(b["x"], b["y"], b["r"])


def show_status(message=""):
    with status:
        status.clear_output()
        print(f"🫧 Score: {score}")
        if message:
            print(message)


def on_click(x, y):
    global score
    for b in bubbles:
        distance = math.hypot(x - b["x"], y - b["y"])
        if distance <= b["r"]:
            bubbles.remove(b)
            bubbles.append(spawn_bubble())
            score += 1
            redraw()
            with status:
                status.clear_output()
                celebrate(f"Pop! Score: {score}")
            return
    show_status("Missed! Try clicking right on a bubble.")


def on_new_game(_):
    global score, bubbles
    score = 0
    bubbles = [spawn_bubble() for _ in range(NUM_BUBBLES)]
    redraw()
    show_status("New game! Pop some bubbles!")


canvas.on_mouse_down(on_click)
new_game_button = widgets.Button(description="🔄 New Game", button_style="warning")
new_game_button.on_click(on_new_game)

bubbles = [spawn_bubble() for _ in range(NUM_BUBBLES)]
redraw()
show_status("Click a bubble to pop it!")
display(canvas)
display(new_game_button, status)


Canvas(height=360, width=480)

Button(button_style='warning', description='🔄 New Game', style=ButtonStyle())

Output()

In [5]:
from ipycanvas import Canvas
import ipywidgets as widgets
from IPython.display import display
import math
import random
import asyncio

WIDTH, HEIGHT = 480, 360
BUBBLE_RADIUS = 22
NUM_BUBBLES = 5
COLORS = ["#e63946", "#f1c40f", "#2a9d8f", "#457b9d", "#e76f51", "#8e44ad"]
GAME_DURATION = 180  # 3 minutes in seconds

canvas = Canvas(width=WIDTH, height=HEIGHT)
bubbles = []
score = 0
time_left = GAME_DURATION
game_active = False
timer_task = None
status = widgets.Output()

def spawn_bubble():
    return {
        "x": random.randint(BUBBLE_RADIUS, WIDTH - BUBBLE_RADIUS),
        "y": random.randint(BUBBLE_RADIUS, HEIGHT - BUBBLE_RADIUS),
        "r": BUBBLE_RADIUS,
        "color": random.choice(COLORS),
    }

def redraw():
    canvas.clear()
    if game_active:
        for b in bubbles:
            canvas.fill_style = b["color"]
            canvas.fill_circle(b["x"], b["y"], b["r"])
    else:
        # Show Game Over screen
        canvas.fill_style = "black"
        canvas.font = "30px sans-serif"
        canvas.fill_text(f"Game Over! Score: {score}", WIDTH // 2 - 140, HEIGHT // 2)

def show_status(message=""):
    with status:
        status.clear_output()
        print(f"🫧 Score: {score} | ⏳ Time: {time_left}s")
        if message:
            print(message)

async def update_timer():
    global time_left, game_active
    while time_left > 0 and game_active:
        await asyncio.sleep(1)
        time_left -= 1
        show_status()
    
    if time_left <= 0 and game_active:
        game_active = False
        redraw()
        show_status("⏰ Time's up! Game Over.")

def on_click(x, y):
    global score
    if not game_active:
        return

    hit = False
    for b in bubbles:
        distance = math.hypot(x - b["x"], y - b["y"])
        if distance <= b["r"]:
            bubbles.remove(b)
            bubbles.append(spawn_bubble())
            score += 1
            hit = True
            break
            
    if hit:
        redraw()
        show_status("Pop!")
    else:
        # Deduct 1 point for a miss
        score -= 1
        show_status("Missed! -1 point.")

def on_new_game(_=None):
    global score, bubbles, time_left, game_active, timer_task
    score = 0
    time_left = GAME_DURATION
    game_active = True
    bubbles = [spawn_bubble() for _ in range(NUM_BUBBLES)]
    
    # Cancel the previous timer if it's still running
    if timer_task is not None and not timer_task.done():
        timer_task.cancel()
    
    # Start a new countdown timer
    timer_task = asyncio.create_task(update_timer())
    
    redraw()
    show_status("New game! Pop some bubbles!")

canvas.on_mouse_down(on_click)
new_game_button = widgets.Button(description="🔄 New Game", button_style="warning")
new_game_button.on_click(on_new_game)

display(canvas)
display(new_game_button, status)

# Start the first game automatically
on_new_game()

Canvas(height=360, width=480)

Button(button_style='warning', description='🔄 New Game', style=ButtonStyle())

Output()

In [6]:
from ipycanvas import Canvas
import ipywidgets as widgets
from IPython.display import display
import math
import random
import asyncio

WIDTH, HEIGHT = 480, 360
BUBBLE_RADIUS = 22
NUM_BUBBLES = 5
COLORS = ["#e63946", "#f1c40f", "#2a9d8f", "#457b9d", "#e76f51", "#8e44ad"]
BOMB_COLOR = "#111111"
GAME_DURATION = 180  # 3 minutes in seconds

canvas = Canvas(width=WIDTH, height=HEIGHT)
bubbles = []
leaderboard = []
score = 0
time_left = GAME_DURATION
game_active = False
timer_task = None
status = widgets.Output()

def spawn_bubble(is_bomb=False):
    return {
        "x": random.randint(BUBBLE_RADIUS, WIDTH - BUBBLE_RADIUS),
        "y": random.randint(BUBBLE_RADIUS, HEIGHT - BUBBLE_RADIUS),
        "r": BUBBLE_RADIUS,
        "color": BOMB_COLOR if is_bomb else random.choice(COLORS),
        "type": "bomb" if is_bomb else "normal"
    }

def redraw():
    canvas.clear()
    if game_active:
        for b in bubbles:
            canvas.fill_style = b["color"]
            canvas.fill_circle(b["x"], b["y"], b["r"])
            # Draw a visual warning for the bomb
            if b["type"] == "bomb":
                canvas.fill_style = "#ff0000"
                canvas.fill_circle(b["x"], b["y"], b["r"] // 2)
    else:
        # Show Game Over screen and Leaderboard
        canvas.fill_style = "black"
        canvas.font = "30px sans-serif"
        canvas.fill_text(f"Game Over! Score: {score}", WIDTH // 2 - 140, HEIGHT // 2 - 60)
        
        canvas.font = "20px sans-serif"
        canvas.fill_text("🏆 Leaderboard (Top 5)", WIDTH // 2 - 100, HEIGHT // 2 - 10)
        for i, s in enumerate(leaderboard):
            canvas.fill_text(f"#{i+1}: {s} pts", WIDTH // 2 - 60, HEIGHT // 2 + 20 + (i * 25))

def show_status(message=""):
    with status:
        status.clear_output()
        print(f"🫧 Score: {score} | ⏳ Time: {time_left}s")
        if message:
            print(message)

def end_game():
    global game_active, leaderboard
    game_active = False
    leaderboard.append(score)
    leaderboard.sort(reverse=True)
    leaderboard = leaderboard[:5]  # Keep only top 5 scores
    redraw()
    show_status("⏰ Time's up! Game Over.")

async def update_timer():
    global time_left, game_active
    while time_left > 0 and game_active:
        await asyncio.sleep(1)
        time_left -= 1
        show_status()
    
    if time_left <= 0 and game_active:
        end_game()

def on_click(x, y):
    global score
    if not game_active:
        return

    hit = False
    for b in bubbles:
        distance = math.hypot(x - b["x"], y - b["y"])
        if distance <= b["r"]:
            hit_type = b["type"]
            bubbles.remove(b)
            # Replace with the exact same type to maintain 1 bomb on screen
            bubbles.append(spawn_bubble(is_bomb=(hit_type == "bomb")))
            
            if hit_type == "bomb":
                score -= 4
                show_status("💥 BOOM! Hit a bomb! -4 points.")
            else:
                score += 1
                show_status("Pop! +1 point.")
                
            hit = True
            break
            
    if hit:
        redraw()
    else:
        # Deduct 1 point for a complete miss
        score -= 1
        show_status("Missed! -1 point.")

def on_new_game(_=None):
    global score, bubbles, time_left, game_active, timer_task
    score = 0
    time_left = GAME_DURATION
    game_active = True
    
    # Spawn normal bubbles and exactly 1 bomb
    bubbles = [spawn_bubble(is_bomb=False) for _ in range(NUM_BUBBLES - 1)]
    bubbles.append(spawn_bubble(is_bomb=True))
    
    # Cancel the previous timer if it's still running
    if timer_task is not None and not timer_task.done():
        timer_task.cancel()
    
    # Start a new countdown timer
    timer_task = asyncio.create_task(update_timer())
    
    redraw()
    show_status("New game! Avoid the black bomb!")

canvas.on_mouse_down(on_click)
new_game_button = widgets.Button(description="🔄 New Game", button_style="warning")
new_game_button.on_click(on_new_game)

display(canvas)
display(new_game_button, status)

# Start the first game automatically
on_new_game()

Canvas(height=360, width=480)

Button(button_style='warning', description='🔄 New Game', style=ButtonStyle())

Output()

## Change the rules

Try editing the game cell above and running it again:

* `NUM_BUBBLES = 5` — more bubbles on screen at once
* `BUBBLE_RADIUS = 22` — bigger bubbles are easier to hit
* Add more colors to the `COLORS` list

## Quick quiz

In [ ]:
quiz(
    question="What does math.hypot(x - b['x'], y - b['y']) calculate?",
    choices=[
        "The bubble's color",
        "The distance between the click and the bubble's center",
        "A random new position",
    ],
    correct_index=1,
    hint="hypot finds the straight-line distance between two points.",
)


Pop pop pop — nice reflexes and nice coding!

In [ ]:
badge("Bubble Popper", emoji="🫧")